# 10 · Qwen3-VL-Embedding-2B — entradas textuales


## 1. Verificar entradas y fijar el perfil
No carga modelos ni crea archivos. Abrir desde `07_RAG` o la raíz del repositorio.

In [1]:
import csv
import hashlib
import json
import sys
import platform
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from time import perf_counter
from uuid import uuid4

VERSION = "QWEN3_VL_2B_TEXTO_BF16_RAG_V1"
REVISION_CODIGO = "QWEN3_VL_2B_TEXTO_IMPL_02_SIN_PAUSAS"
MODELO = "Qwen/Qwen3-VL-Embedding-2B"
REVISION = "9f2f7e710d6d81056aa5c0a4f04764fec6bb7bda"
LOTE = 1
DIMENSION = 2048
LIMITE_TOKENS = 32768
NOMBRE_PROMPT = "default"
NOMBRE_PROMPT_DOCUMENTO = "default"
PREFIJO = "Represent the user's input."
PREFIJO_DOCUMENTO = "Represent the user's input."
ATENCION = "sdpa"  # Perfil PyTorch local: no requiere instalar FlashAttention.
SHA_ESPERADO = "74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b"
# Valor HISTÓRICO conservado exclusivamente en PERFIL para verificar la firma
# del NPZ creado por la libreta anterior; YA NO ACTIVA NINGUNA PAUSA.
PAUSA_TRAS_LLAMADA_S = 30.0
# Control de ejecución: no existen umbrales de duración para el piloto/catálogo.
CONTROL_PAUSA_POR_TIEMPO = False


def sha(ruta):
    return hashlib.sha256(ruta.read_bytes()).hexdigest()


def ahora():
    return datetime.now(timezone.utc).isoformat()


REPO = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent)
             if (p / "06_grafos/catalogo_articulos.csv").is_file()
             and (p / "07_RAG/auditoria_catalogo.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre la libreta desde 07_RAG o desde la raíz del repositorio.")
BASE = REPO / "07_RAG"
CATALOGO = REPO / "06_grafos/catalogo_articulos.csv"
AUDITORIA = BASE / "auditoria_catalogo.json"
CONSULTAS = BASE / "consultas_para_revision.csv"
VECTORES = BASE / "qwen3_vl_2b_text_vectores_v1.npz"
INDICE = BASE / "qwen3_vl_2b_text_indice_v1.faiss"

SHA_CATALOGO = sha(CATALOGO)
SHA_AUDITORIA = sha(AUDITORIA)
auditoria = json.loads(AUDITORIA.read_text(encoding="utf-8"))
if SHA_CATALOGO != SHA_ESPERADO or SHA_CATALOGO != auditoria["fuentes"]["catalogo"]["sha256_bytes"]:
    raise ValueError("El catálogo no coincide con la versión auditada. No continuar.")
if auditoria["resumen"]["incidencias_bloqueantes"] != 0:
    raise ValueError("Hay incidencias bloqueantes en la auditoría.")
with CATALOGO.open(encoding="utf-8-sig", newline="") as f:
    articulos = list(csv.DictReader(f))
article_ids = [r["article_id"] for r in articulos]
resumenes = [r["abstract"] for r in articulos]
if len(article_ids) != 519 or len(set(article_ids)) != 519 or len(set(resumenes)) != 514:
    raise ValueError("Cambió la estructura del catálogo.")
if not all(t.strip() for t in resumenes):
    raise ValueError("Hay resúmenes vacíos; no se sustituyen con texto generado.")
with CONSULTAS.open(encoding="utf-8-sig", newline="") as f:
    consultas = list(csv.DictReader(f))
por_query = {r["query_id"]: r for r in consultas}
if len(por_query) != len(consultas) or not {"N01-ES", "N01-EN"}.issubset(por_query):
    raise ValueError("Revisar identificadores de las consultas; no se inventan consultas sustitutas.")
SHA_CONSULTAS = sha(CONSULTAS)

PERFIL = {
    "version": VERSION, "modelo": MODELO, "revision": REVISION,
    "dispositivo": "cuda", "dtype_pesos_solicitado": "bfloat16", "dimensiones": DIMENSION,
    "batch_size": LOTE, "limite_tokens": LIMITE_TOKENS, "campo": "abstract",
    "prefijo_documento": PREFIJO_DOCUMENTO, "nombre_prompt_documento": NOMBRE_PROMPT_DOCUMENTO,
    "nombre_prompt_consulta": NOMBRE_PROMPT,
    "prefijo_consulta": PREFIJO, "pooling": "lasttoken", "include_prompt": True,
    "atencion_solicitada": ATENCION, "arquitectura_esperada": "qwen3_vl",
    "trust_remote_code": False, "reduccion_dimensional": None,
    "config_kwargs": {},
    "modalidad_entrada": "solo_texto; se conserva arquitectura VL y se registran pesos visuales", "titulos_incorporados": False,
    "processing_kwargs": {"text": {"truncation": False, "padding": True},
                          "chat_template": {"add_generation_prompt": True}},
    "tipo_instruccion": "mensaje de sistema via plantilla oficial, no prefijo literal concatenado",
    "unpad_inputs_fuente": False,
    "torch_compile_invocado": False,
    "license_model_card": "Apache-2.0",
    "normalizacion": "L2 final float32", "indice": "IndexFlatIP CPU",
    "orden_lotes": "consecutivos en el orden del catalogo; orden interno de Sentence Transformers",
    # Campo histórico firmado: NO es una restricción activa en esta versión.
    "pausa_operativa_s": PAUSA_TRAS_LLAMADA_S,
}
FIRMA = hashlib.sha256(json.dumps(
    {"perfil": PERFIL, "catalogo": SHA_CATALOGO}, sort_keys=True,
    ensure_ascii=False).encode("utf-8")).hexdigest()

print("Protocolo:", VERSION, "| Implementación:", REVISION_CODIGO)
print("Modelo:", MODELO, "| Revisión:", REVISION)
print("Catálogo verificado:", SHA_CATALOGO)
print("Artículos:", len(article_ids), "| Resúmenes distintos:", len(set(resumenes)))
print("Consultas:", len(consultas), "| Revisión humana:",
      dict(Counter(r.get("revision_humana", "no registrada") for r in consultas)))
print("Salida:", BASE)
print("Esta celda NO carga modelos ni crea archivos.")

Protocolo: QWEN3_VL_2B_TEXTO_BF16_RAG_V1 | Implementación: QWEN3_VL_2B_TEXTO_IMPL_02_SIN_PAUSAS
Modelo: Qwen/Qwen3-VL-Embedding-2B | Revisión: 9f2f7e710d6d81056aa5c0a4f04764fec6bb7bda
Catálogo verificado: 74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b
Artículos: 519 | Resúmenes distintos: 514
Consultas: 24 | Revisión humana: {'pendiente': 24}
Salida: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG
Esta celda NO carga modelos ni crea archivos.


## 2. Comprobar entorno y definir funciones
No carga el modelo. No instala ni actualiza bibliotecas. La siguiente celda activa el piloto.

In [2]:
import numpy as np
import torch
import faiss
import sentence_transformers
import transformers
import huggingface_hub
from importlib.metadata import version as version_paquete, PackageNotFoundError
from sentence_transformers import SentenceTransformer
from packaging.version import Version

ENTORNO = {
    "python": sys.version.split()[0], "sistema": platform.platform(),
    "numpy": np.__version__, "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "sentence_transformers": sentence_transformers.__version__,
    "transformers": transformers.__version__, "huggingface_hub": huggingface_hub.__version__,
    "faiss": getattr(faiss, "__version__", "no_registrada"),
}
if not torch.cuda.is_available():
    raise RuntimeError("Este perfil requiere CUDA. No se cambia a CPU automáticamente.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("CUDA no declara soporte bfloat16. No se cambia a float16/CPU.")
if Version(sentence_transformers.__version__) < Version("5.4.0"):
    raise RuntimeError("Este perfil requiere Sentence Transformers >=5.4.0. No se actualiza automáticamente.")
# Comprobar que Transformers conoce la arquitectura antes de descargar pesos.
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
if "qwen3_vl" not in CONFIG_MAPPING:
    raise RuntimeError("Esta instalación de Transformers no reconoce qwen3_vl; conserva el error.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("La GPU no reporta soporte bfloat16; no se cambia precisión automáticamente.")
for paquete in ("torchvision", "Pillow", "safetensors"):
    try:
        ENTORNO[paquete] = version_paquete(paquete)
    except PackageNotFoundError:
        ENTORNO[paquete] = "no_instalado"
ENTORNO["matmul_precision"] = torch.get_float32_matmul_precision()
ENTORNO["tf32_matmul_permitido"] = bool(torch.backends.cuda.matmul.allow_tf32)
ENTORNO["tf32_cudnn_permitido"] = bool(torch.backends.cudnn.allow_tf32)
try:
    import torchvision
except (ImportError, RuntimeError) as exc:
    raise ImportError("El procesador de Qwen-VL requiere torchvision compatible. No actualices todo el entorno; revisa la instalación que ya funcionó con EmbeddingGemma 2.") from exc
ENTORNO["gpu"] = torch.cuda.get_device_name(0)
for nombre, valor in ENTORNO.items():
    print(nombre + ":", valor)
libre, total = torch.cuda.mem_get_info()
print("VRAM libre / total (GiB):", round(libre / 2**30, 2), "/", round(total / 2**30, 2))


def comprobar_entradas():
    if sha(CATALOGO) != SHA_CATALOGO or sha(AUDITORIA) != SHA_AUDITORIA:
        raise ValueError("Cambió una entrada durante la sesión. No continuar.")
    if sha(CONSULTAS) != SHA_CONSULTAS:
        raise ValueError("Cambió el CSV de consultas; vuelve a ejecutar la celda 1.")


def memoria():
    libre, total = torch.cuda.mem_get_info()
    return {
        "libre_bytes": int(libre), "total_bytes": int(total),
        "asignada_pytorch_bytes": int(torch.cuda.memory_allocated()),
        "reservada_pytorch_bytes": int(torch.cuda.memory_reserved()),
        "pico_asignada_pytorch_bytes": int(torch.cuda.max_memory_allocated()),
        "pico_reservada_pytorch_bytes": int(torch.cuda.max_memory_reserved()),
        "alcance": "PyTorch del proceso y lecturas puntuales CUDA; no mide RAM del sistema",
    }


def cargar_estado():
    if VECTORES.with_name(VECTORES.name + ".tmp").exists():
        raise FileExistsError("Existe un NPZ .tmp de una escritura interrumpida. No se borra ni se reemplaza automáticamente; conservar para revisión.")
    if not VECTORES.exists():
        if INDICE.exists():
            raise FileExistsError("Existe un índice sin NPZ asociado; revisar sin borrar nada.")
        meta = {
            "firma": FIRMA, "perfil": PERFIL, "creado_utc": ahora(), "estado": "nuevo",
            "sha256_catalogo": SHA_CATALOGO, "sha256_auditoria_inicial": SHA_AUDITORIA,
            "pilotos": [], "cargas": [], "sesiones_catalogo": [], "lotes": [],
            "errores": [], "busquedas": [], "entorno_vectorizacion": None,
        }
        return meta, np.empty((0, DIMENSION), dtype=np.float32)
    with np.load(VECTORES, allow_pickle=False) as z:
        meta = json.loads(z["metadata_json"].item())
        matriz = z["vectores"].copy()
        ids = z["article_ids"].tolist()
        if (meta["firma"] != FIRMA or meta["perfil"] != PERFIL
                or z["article_ids_catalogo"].tolist() != article_ids
                or z["modelo"].item() != MODELO or z["modelo_revision"].item() != REVISION
                or z["sha256_catalogo"].item() != SHA_CATALOGO):
            raise ValueError("El NPZ pertenece a otro perfil, catálogo o versión. No se sobrescribe.")
        if z["sha256_matriz"].item() != hashlib.sha256(matriz.tobytes()).hexdigest():
            raise ValueError("La matriz guardada no coincide con su huella.")
    if (matriz.dtype != np.float32 or matriz.ndim != 2 or matriz.shape[1] != DIMENSION
            or len(matriz) > len(article_ids) or ids != article_ids[:len(matriz)]):
        raise ValueError("Forma u orden de las filas del NPZ incorrectos.")
    if not np.isfinite(matriz).all():
        raise ValueError("Hay valores no finitos en el NPZ.")
    if len(matriz) and not np.allclose(np.linalg.norm(matriz, axis=1), 1, rtol=0, atol=1e-5):
        raise ValueError("Las normas guardadas no son unitarias.")
    cursor = 0
    for registro in meta["lotes"]:
        fin = registro["fin_fila_exclusivo"]
        if (registro["inicio_fila"] != cursor or not cursor < fin <= len(matriz)
                or registro["article_ids"] != article_ids[cursor:fin]):
            raise ValueError("El historial de lotes no corresponde a la matriz guardada.")
        cursor = fin
    if cursor != len(matriz):
        raise ValueError("Hay vectores sin lote registrado o lotes repetidos.")
    if meta["estado"] == "completo" and len(matriz) != len(article_ids):
        raise ValueError("Un checkpoint incompleto no puede declararse completo.")
    if INDICE.exists() and len(matriz) != len(article_ids):
        raise ValueError("Un índice existe junto a un NPZ incompleto. Revisar sin sobrescribir.")
    return meta, matriz


def registrar_ejecucion_sin_pausas(meta):
    # Registra el cambio operativo SIN cambiar la firma del perfil de embeddings.
    historial = meta.setdefault("historial_control_temporal", [])
    if not any(x.get("revision_codigo") == REVISION_CODIGO for x in historial):
        historial.append({
            "fecha_utc": ahora(), "revision_codigo": REVISION_CODIGO,
            "detencion_automatica_por_duracion": False,
            "perfil_firmado_compatible_con_version_anterior": True,
        })


def guardar_estado(meta, matriz):
    # Sustituye solo el avance de ESTE perfil. Los resultados de otros modelos no se tocan.
    meta["actualizado_utc"] = ahora()
    temporal = VECTORES.with_name(VECTORES.name + ".tmp")
    if temporal.exists():
        raise FileExistsError("Hay un .tmp de una interrupción de escritura; conservarlo y revisar.")
    try:
        with temporal.open("xb") as f:
            np.savez_compressed(
                f, vectores=matriz, article_ids=np.asarray(article_ids[:len(matriz)]),
                article_ids_catalogo=np.asarray(article_ids),
                modelo=np.asarray(MODELO), modelo_revision=np.asarray(REVISION),
                sha256_catalogo=np.asarray(SHA_CATALOGO), campo=np.asarray("abstract"),
                completo=np.asarray(meta["estado"] == "completo"), estado=np.asarray(meta["estado"]),
                sha256_matriz=np.asarray(hashlib.sha256(matriz.tobytes()).hexdigest()),
                tiempo_vectorizacion_s=np.asarray(sum(r["encode_s"] for r in meta["lotes"])),
                tokens_por_articulo=np.asarray(meta.get("modelo_runtime", {}).get("tokens_por_articulo", []), dtype=np.int64),
                metadata_json=np.asarray(json.dumps(meta, ensure_ascii=False, allow_nan=False)),
            )
        temporal.replace(VECTORES)
    except BaseException:
        print("Escritura interrumpida. Se conserva el NPZ anterior y, si existe, su .tmp para revisión.")
        raise


def registrar_error(meta, matriz, etapa, error):
    meta["errores"].append({"fecha_utc": ahora(), "etapa": etapa,
                            "tipo": type(error).__name__, "mensaje": str(error),
                            "revision_codigo": REVISION_CODIGO})
    if meta["estado"] != "completo":
        meta["estado"] = "interrumpido_o_error"
    try:
        guardar_estado(meta, matriz)
    except FileExistsError as e:
        print("Se conserva el último checkpoint y su temporal. No se sobrescribe:", e)
        raise


def verificar_pooling_compatible(modelo):
    """Comprobar el módulo cargado sin cambiar sus parámetros ni su forward."""
    pooling = modelo[1]  # La revisión fijada de modules.json coloca Pooling en el índice 1.
    if type(pooling).__name__ != "Pooling":
        raise ValueError("El módulo 1 no es Pooling; no se fuerza otra arquitectura.")
    obtener_config = getattr(pooling, "get_config_dict", None)
    if not callable(obtener_config):
        raise ValueError("No se puede consultar la configuración efectiva de Pooling.")
    config_pooling = obtener_config()
    if not isinstance(config_pooling, dict):
        raise ValueError("La configuración efectiva de Pooling no es un diccionario.")

    # API actual: un modo como cadena, o varios modos como lista/tupla.
    # Los modos adicionales no son admisibles para esta revisión de EmbeddingGemma 2.
    banderas_legacy = {
        "pooling_mode_cls_token": "cls",
        "pooling_mode_max_tokens": "max",
        "pooling_mode_mean_tokens": "mean",
        "pooling_mode_mean_sqrt_len_tokens": "mean_sqrt_len_tokens",
        "pooling_mode_weightedmean_tokens": "weightedmean",
        "pooling_mode_lasttoken": "lasttoken",
    }
    modo = config_pooling.get("pooling_mode")
    if isinstance(modo, str):
        modos = (modo,)
        formato = "pooling_mode"
    elif isinstance(modo, (list, tuple)) and all(isinstance(x, str) for x in modo):
        modos = tuple(modo)
        formato = "pooling_mode"
    elif modo is None and any(k in config_pooling for k in banderas_legacy):
        if any(config_pooling[k] not in (True, False) for k in banderas_legacy if k in config_pooling):
            raise ValueError("Las banderas antiguas de Pooling no son booleanas.")
        modos = tuple(v for k, v in banderas_legacy.items() if config_pooling.get(k, False))
        formato = "banderas_legacy"
    else:
        raise ValueError(f"Formato de Pooling no reconocido: {config_pooling!r}")

    # No admitir declaraciones contradictorias si una implementación expone ambos formatos.
    if formato == "pooling_mode":
        for clave, nombre in banderas_legacy.items():
            if clave in config_pooling and config_pooling[clave] != (nombre in modos):
                raise ValueError("La configuración mezcla declaraciones de Pooling contradictorias.")
    if modos != ("lasttoken",):
        raise ValueError(f"Pooling realmente distinto del esperado: {modos!r}. No se modifica.")
    if config_pooling.get("include_prompt") is not True:
        raise ValueError("Este perfil conserva include_prompt=True; no se modifica el módulo.")
    dimension_pooling = config_pooling.get(
        "embedding_dimension", config_pooling.get("word_embedding_dimension")
    )
    if dimension_pooling != DIMENSION:
        raise ValueError(f"Dimensión de Pooling inesperada: {dimension_pooling!r}.")
    return {
        "modos": list(modos), "formato": formato,
        "clase": f"{type(pooling).__module__}.{type(pooling).__name__}",
        "configuracion": json.loads(json.dumps(config_pooling, ensure_ascii=False, allow_nan=False)),
    }



def leer_configuraciones_fuente():
    """Fijar y registrar la configuración original: no ejecutar scripts del repositorio."""
    nombres = ["config.json", "config_sentence_transformers.json", "modules.json",
               "1_Pooling/config.json", "sentence_bert_config.json", "tokenizer_config.json",
               "chat_template.jinja", "preprocessor_config.json", "video_preprocessor_config.json"]
    datos, registro = {}, {}
    for nombre in nombres:
        ruta = Path(huggingface_hub.hf_hub_download(repo_id=MODELO, filename=nombre, revision=REVISION))
        contenido = ruta.read_bytes()
        registro[nombre] = {"revision": REVISION, "sha256": hashlib.sha256(contenido).hexdigest(),
                            "bytes": len(contenido)}
        if nombre.endswith(".json"):
            datos[nombre] = json.loads(contenido.decode("utf-8"))
    config = datos["config.json"]
    texto = config.get("text_config", {})
    if (config.get("model_type") != "qwen3_vl" or texto.get("model_type") != "qwen3_vl_text"
            or texto.get("hidden_size") != DIMENSION
            or config.get("dtype", config.get("torch_dtype")) != "bfloat16"):
        raise ValueError("La configuración oficial no coincide con Qwen3-VL-Embedding-2B.")
    prompts = datos["config_sentence_transformers.json"]
    if (prompts.get("default_prompt_name") != "default"
            or prompts.get("prompts", {}).get("default") != PREFIJO):
        raise ValueError("La instrucción predeterminada difiere de la documentada; no se sustituye.")
    if [x["type"].split(".")[-1] for x in datos["modules.json"]] != ["Transformer", "Pooling", "Normalize"]:
        raise ValueError("La estructura oficial no es Transformer + Pooling + Normalize.")
    pooling = datos["1_Pooling/config.json"]
    if (pooling.get("pooling_mode") != "lasttoken" or pooling.get("include_prompt") is not True
            or pooling.get("embedding_dimension") != DIMENSION):
        raise ValueError("No coincide la agregación lasttoken de 2048 dimensiones.")
    st = datos["sentence_bert_config.json"]
    if (st.get("transformer_task") != "feature-extraction" or st.get("unpad_inputs") is not False
            or st.get("modality_config", {}).get("message", {}).get("format") != "structured"
            or st.get("processing_kwargs", {}).get("chat_template", {}).get("add_generation_prompt") is not True):
        raise ValueError("La plantilla estructurada / unpad_inputs no coincide con la revisión fijada.")
    return registro


PROCESAMIENTO = {
    "text": {"padding": True, "truncation": False},
    "chat_template": {"add_generation_prompt": True},
}


def entradas_texto(textos):
    # Un dict con clave text evita interpretar una cadena como ruta/URL de una imagen.
    if not textos or not all(isinstance(t, str) and t.strip() for t in textos):
        raise ValueError("Se requieren textos no vacíos; no se sustituyen entradas.")
    return [{"text": t} for t in textos]


def contar_tokens(modelo, textos, instruccion):
    """Cuenta la entrada REAL de ST con plantilla de chat, no solo el resumen desnudo."""
    longitudes = []
    for ini in range(0, len(textos), 16):
        tramo = textos[ini:ini+16]
        preparados = modelo.preprocess(
            entradas_texto(tramo), prompt=instruccion, processing_kwargs=PROCESAMIENTO)
        for clave in ("pixel_values", "pixel_values_videos", "image_grid_thw", "video_grid_thw"):
            if preparados.get(clave) is not None:
                raise ValueError("Se preparó contenido visual; esta condición acepta únicamente texto.")
        ids, mascara = preparados.get("input_ids"), preparados.get("attention_mask")
        if (ids is None or mascara is None or ids.ndim != 2
                or mascara.shape != ids.shape or ids.shape[0] != len(tramo)):
            raise ValueError("El preprocesador no produjo una máscara textual válida.")
        conteos = mascara.sum(dim=1).detach().cpu().tolist()
        if any(n <= 0 or int(n) != n for n in conteos):
            raise ValueError("Conteo de tokens no válido.")
        longitudes.extend(int(n) for n in conteos)
    return longitudes


def verificar_precision_arquitectura(modelo):
    base = modelo[0].auto_model
    conf = base.config
    texto = getattr(conf, "text_config", None)
    if (getattr(conf, "model_type", None) != "qwen3_vl"
            or getattr(texto, "model_type", None) != "qwen3_vl_text"
            or getattr(texto, "hidden_size", None) != DIMENSION):
        raise ValueError("Arquitectura o dimensión textual no coincidente.")
    por_dtype, total, visual = {}, 0, 0
    for nombre, parametro in modelo.named_parameters():
        if parametro.device.type != "cuda":
            raise ValueError("Se detectaron pesos fuera de CUDA; no hay reparto automático con CPU.")
        if parametro.dtype not in (torch.bfloat16, torch.float32):
            raise ValueError("Dtype inesperado; no se cambia a float16 ni se cuantizan pesos.")
        cantidad = int(parametro.numel())
        total += cantidad
        por_dtype[str(parametro.dtype)] = por_dtype.get(str(parametro.dtype), 0) + cantidad
        if any(pieza in nombre.lower() for pieza in ("visual", "vision")):
            visual += cantidad
    if not por_dtype.get("torch.bfloat16", 0):
        raise ValueError("No se cargaron los pesos BF16 solicitados.")
    raiz = getattr(conf, "_attn_implementation", None)
    atencion_texto = getattr(texto, "_attn_implementation", None)
    efectiva = atencion_texto or (raiz.get("text_config") if isinstance(raiz, dict) else raiz)
    if efectiva != ATENCION:
        raise ValueError(f"La atención textual reportada no es SDPA: {efectiva!r}.")
    return {"parametros_cargados": total, "parametros_por_dtype": por_dtype,
            "parametros_con_nombre_visual": visual,
            "vision_config_presente": getattr(conf, "vision_config", None) is not None,
            "atencion_texto_reportada": atencion_texto, "atencion_raiz_reportada": raiz,
            "nota": "Entrada exclusivamente textual. No se elimina la torre visual; no se asume memoria de un modelo solo textual."}


def asegurar_modelo(meta):
    global codificador_qvl, runtime_qvl, carga_qvl, firma_cargada
    if "codificador_qvl" not in globals():
        print("Cargando Qwen3-VL-Embedding-2B; no se envían imágenes ni vídeo...", flush=True)
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        memoria_previa = memoria()
        inicio = perf_counter()
        fuentes = leer_configuraciones_fuente()
        m = SentenceTransformer(
            MODELO, revision=REVISION, device="cuda", trust_remote_code=False,
            model_kwargs={"dtype": torch.bfloat16, "attn_implementation": ATENCION,
                          "use_safetensors": True},
        )
        m.eval()
        torch.cuda.synchronize()
        carga_s = perf_counter() - inicio
        verificacion = verificar_precision_arquitectura(m)
        obtener_dimension = getattr(m, "get_embedding_dimension", None)
        if not callable(obtener_dimension):
            obtener_dimension = getattr(m, "get_sentence_embedding_dimension", None)
        if not callable(obtener_dimension) or obtener_dimension() != DIMENSION:
            raise ValueError("Se esperaban 2048 componentes sin recorte.")
        if m.default_prompt_name != "default" or m.prompts.get("default") != PREFIJO:
            raise ValueError("Cambió la instrucción oficial; no se reutilizan los prefijos de Qwen3 textual.")
        if len(m) != 3 or type(m[2]).__name__ != "Normalize":
            raise ValueError("Estructura de módulos inesperada.")
        if getattr(m, "truncate_dim", None) is not None or getattr(m[0], "do_lower_case", False):
            raise ValueError("No se permite reducción dimensional ni conversión a minúsculas.")
        if getattr(m[0], "unpad_inputs", None) is not False:
            raise ValueError("Qwen-VL requiere unpad_inputs=False en esta revisión.")
        pooling = verificar_pooling_compatible(m)
        print("Pooling verificado:", pooling["modos"], "| include_prompt:", pooling["configuracion"]["include_prompt"], flush=True)
        original = m.max_seq_length
        if original is not None and int(original) <= 0:
            raise ValueError("Límite de entrada no válido.")
        m.max_seq_length = min(int(original), LIMITE_TOKENS) if original is not None else LIMITE_TOKENS
        longitudes = contar_tokens(m, resumenes, PREFIJO_DOCUMENTO)
        excedidos = [article_ids[i] for i, n in enumerate(longitudes) if n > m.max_seq_length]
        if excedidos:
            raise ValueError("Entradas demasiado largas; no se recortan: " + ", ".join(excedidos))
        conf = m[0].auto_model.config
        commit = getattr(conf, "_commit_hash", None)
        if commit is not None and commit != REVISION:
            raise ValueError("La revisión reportada difiere de la solicitada.")
        # Trazabilidad de la entrada real: token ids y texto de una muestra preparados por ST.
        muestra = m.preprocess(entradas_texto([resumenes[0]]), prompt=PREFIJO_DOCUMENTO,
                               processing_kwargs=PROCESAMIENTO)
        ids_muestra = muestra["input_ids"][0][muestra["attention_mask"][0].bool()].detach().cpu().tolist()
        procesador = getattr(m[0], "processor", None)
        runtime_qvl = {
            "dtype_solicitado": "bfloat16", "revision_solicitada": REVISION,
            "revision_reportada": commit, "fuentes_oficiales": fuentes,
            "dimension": DIMENSION, "trust_remote_code": False,
            "max_seq_length_original": int(original) if original is not None else None,
            "max_seq_length": int(m.max_seq_length), "tokens_por_articulo": longitudes,
            "articulos_truncados": 0, "modalidad_entrada": "text",
            "carga_arquitectura": verificacion, "pooling_verificado": pooling,
            "instruccion_consulta": PREFIJO, "instruccion_documento": PREFIJO_DOCUMENTO,
            "nombre_prompt_consulta": "default", "nombre_prompt_documento": "default",
            "instruccion_como": "system message por la plantilla oficial de ST; no concatenación manual",
            "processing_kwargs": PROCESAMIENTO, "modulos": [type(x).__name__ for x in m],
            "preprocesador_clase": type(procesador).__name__ if procesador is not None else None,
            "ids_tokenizados_muestra_article_id_1": ids_muestra,
            "tokens_decodificados_muestra": m.tokenizer.decode(ids_muestra, skip_special_tokens=False),
            "torch_compile_invocado": False,
        }
        carga_qvl = {"id": str(uuid4()), "fecha_utc": ahora(), "tiempo_carga_s": carga_s,
                     "revision_codigo": REVISION_CODIGO, "entorno": ENTORNO,
                     "memoria_antes": memoria_previa, "memoria_despues": memoria()}
        codificador_qvl, firma_cargada = m, FIRMA
    elif firma_cargada != FIRMA:
        raise ValueError("Hay otro perfil cargado; reinicia el kernel.")
    anterior = meta.get("modelo_runtime")
    if anterior is not None and anterior != runtime_qvl:
        raise ValueError("Cambió el procesamiento efectivo. No se mezclan vectores de distintos perfiles.")
    meta["modelo_runtime"] = runtime_qvl
    if not any(c["id"] == carga_qvl["id"] for c in meta["cargas"]):
        meta["cargas"].append(carga_qvl)
    print("Modelo: cuda | BF16 solicitado | Carga:", round(carga_qvl["tiempo_carga_s"], 3), "s")
    print("Parámetros por dtype:", runtime_qvl["carga_arquitectura"]["parametros_por_dtype"])
    print("Parámetros con nombre visual conservados:", runtime_qvl["carga_arquitectura"]["parametros_con_nombre_visual"])
    print("Tokens de documentos mín./máx. (incluida plantilla):", min(runtime_qvl["tokens_por_articulo"]),
          max(runtime_qvl["tokens_por_articulo"]), "| Truncados: 0")
    print("Instrucción oficial para ambos tipos de texto:", repr(PREFIJO))


def codificar(textos, consulta=False, lote=LOTE):
    # La instrucción DEFAULT oficial se inserta como system por ST, nunca dos veces.
    longitudes = contar_tokens(codificador_qvl, textos, PREFIJO)
    if any(n > codificador_qvl.max_seq_length for n in longitudes):
        raise ValueError("Entrada demasiado larga; no se codifica con truncamiento.")
    torch.cuda.synchronize()
    inicio = perf_counter()
    with torch.inference_mode():
        salida = codificador_qvl.encode(
            entradas_texto(textos), prompt_name="default", batch_size=lote,
            show_progress_bar=False, normalize_embeddings=True, convert_to_numpy=True,
            precision="float32", processing_kwargs=PROCESAMIENTO,
        )
    torch.cuda.synchronize()
    encode_s = perf_counter() - inicio
    inicio_l2 = perf_counter()
    v = np.array(salida, dtype=np.float32, order="C", copy=True)
    if v.shape != (len(textos), DIMENSION) or not np.isfinite(v).all():
        raise ValueError("Forma o valores de salida incorrectos.")
    antes = np.linalg.norm(v, axis=1)
    if not np.isfinite(antes).all() or np.any(antes <= 0):
        raise ValueError("Vector nulo o norma no finita.")
    faiss.normalize_L2(v)
    despues = np.linalg.norm(v, axis=1)
    if not np.isfinite(v).all() or not np.allclose(despues, 1, rtol=0, atol=1e-5):
        raise ValueError("Falló la normalización L2 final en float32.")
    return v, {"encode_s": encode_s, "normalizacion_s": perf_counter()-inicio_l2,
               "tokens": longitudes, "dtype_encode": str(np.asarray(salida).dtype),
               "tipo_entrada": "query" if consulta else "document",
               "instruccion_sistema": PREFIJO,
               "normas_antes_min": float(antes.min()), "normas_antes_max": float(antes.max()),
               "desviacion_despues_max": float(np.max(np.abs(despues-1)))}


def finalizar_indice(meta, matriz):
    if len(matriz) != len(article_ids):
        raise ValueError("No se permite crear un índice incompleto.")
    if not INDICE.exists():
        inicio = perf_counter()
        idx = faiss.IndexFlatIP(DIMENSION)
        idx.add(matriz)
        meta["tiempo_construccion_indice_s"] = perf_counter() - inicio
        # Archivo temporal para no dejar un índice final incompleto tras una interrupción.
        temporal_indice = INDICE.with_name(INDICE.name + ".tmp")
        if temporal_indice.exists():
            raise FileExistsError("Hay un índice .tmp pendiente de revisar; no se reemplaza.")
        with temporal_indice.open("xb") as f:
            f.write(faiss.serialize_index(idx).tobytes())
        if INDICE.exists():
            raise FileExistsError("El índice apareció durante la escritura; no se reemplaza.")
        temporal_indice.replace(INDICE)
    idx = faiss.read_index(str(INDICE))
    if (idx.ntotal != len(article_ids) or idx.d != DIMENSION
            or idx.metric_type != faiss.METRIC_INNER_PRODUCT
            or not np.array_equal(idx.reconstruct_n(0, len(matriz)), matriz)):
        raise ValueError("El índice existente no coincide con los vectores. No se sobrescribe.")
    meta["estado"] = "completo"
    meta["sha256_indice"] = sha(INDICE)
    guardar_estado(meta, matriz)
    cargar_estado()  # Comprobar también la relectura del NPZ.
    print("Artículos indexados:", idx.ntotal, "| Dimensión:", idx.d)
    print("Vectorización (suma de llamadas, sin piloto):", round(sum(r["encode_s"] for r in meta["lotes"]), 3), "s")
    print("Guardados:", VECTORES.name, "y", INDICE.name)
    print("SHA-256 NPZ:", sha(VECTORES))
    print("SHA-256 FAISS:", sha(INDICE))
    print("No se calcularon métricas de calidad de recuperación.")

print("Bibliotecas listas. Ningún modelo está cargado todavía.")
print("Perfil: BF16 + SDPA + lote 1. Solo entradas textuales; se conserva la arquitectura VL.")
print("No se instala FlashAttention ni se recorta el texto; primero ejecutar el piloto.")

W1108 02:49:40.469000 22568 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


python: 3.11.16
sistema: Windows-10-10.0.19045-SP0
numpy: 2.3.5
torch: 2.11.0+cu128
cuda_runtime: 12.8
sentence_transformers: 6.2.0
transformers: 5.22.2
huggingface_hub: 2.4.0
faiss: 1.15.1
torchvision: 0.26.0+cu128
Pillow: 12.3.0
safetensors: 0.8.0
matmul_precision: highest
tf32_matmul_permitido: False
tf32_cudnn_permitido: True
gpu: NVIDIA GeForce RTX 4070 SUPER
VRAM libre / total (GiB): 10.81 / 11.99
Bibliotecas listas. Ningún modelo está cargado todavía.
Perfil: BF16 + SDPA + lote 1. Solo entradas textuales; se conserva la arquitectura VL.
No se instala FlashAttention ni se recorta el texto; primero ejecutar el piloto.


## 3. Piloto pequeño
Activar solo `EJECUTAR_PILOTO=True`

In [ ]:
EJECUTAR_PILOTO = True
REPETIR_PILOTO = False

if not EJECUTAR_PILOTO:
    print("Preparado. Cambia EJECUTAR_PILOTO=True para la prueba pequeña, no para los 519 artículos.")
else:
    comprobar_entradas()
    meta, matriz = cargar_estado()
    if meta["estado"] == "completo" or len(matriz):
        print("Ya hay vectores del catálogo; no se repite el piloto. Consulta el registro existente.")
    elif meta["pilotos"] and meta["pilotos"][-1]["estado"] == "completo" and not REPETIR_PILOTO:
        print("Piloto ya completo; no se repite. Puedes revisar sus tiempos y continuar con la celda 4.")
    else:
        # Reanudar el piloto detenido por el antiguo umbral de 30 s,
        # SIN repetir los ensayos ya terminados y guardados.
        previo = meta["pilotos"][-1] if meta["pilotos"] else None
        if (previo is not None and previo.get("estado") == "pausa_operativa"
                and not REPETIR_PILOTO):
            if (previo.get("entorno") != ENTORNO
                    or previo.get("sha256_consultas") != SHA_CONSULTAS):
                raise ValueError("El piloto pausado se hizo con otro entorno/consultas. No reanudar.")
            piloto = previo
            piloto.setdefault("reanudaciones", []).append({
                "fecha_utc": ahora(), "revision_codigo": REVISION_CODIGO,
                "sin_pausa_por_tiempo": True,
            })
            piloto["estado"] = "en_curso"
            piloto.pop("fin_utc", None)
            print("Reanudando piloto pausado: ensayos guardados:", len(piloto["ensayos"]))
        else:
            piloto = {"id": str(uuid4()), "inicio_utc": ahora(), "estado": "en_curso",
                      "entorno": ENTORNO, "sha256_consultas": SHA_CONSULTAS, "ensayos": [],
                      "revision_codigo": REVISION_CODIGO}
            meta["pilotos"].append(piloto)
        registrar_ejecucion_sin_pausas(meta)
        meta["estado"] = "diagnostico"
        guardar_estado(meta, matriz)
        try:
            asegurar_modelo(meta)
            guardar_estado(meta, matriz)
            longitudes = runtime_qvl["tokens_por_articulo"]
            orden = sorted(range(len(resumenes)), key=lambda i: (longitudes[i], i))
            tareas = [
                ("corto_primera_llamada", [orden[0]], False),
                ("corto_repeticion", [orden[0]], False),
                ("mediano", [orden[len(orden)//2]], False),
                ("mas_largo", [orden[-1]], False),
                ("primeros_cuatro_secuenciales_lote1", [0, 1, 2, 3], False),
                ("N01-ES", [], True), ("N01-EN", [], True),
            ]
            # Asegurar que el historial guardado corresponde al prefijo exacto
            # de este mismo protocolo. Así nunca se omiten ensayos diferentes.
            esperados = [nombre for nombre, _, _ in tareas]
            previos = [r.get("prueba") for r in piloto["ensayos"]]
            if previos != esperados[:len(previos)]:
                raise ValueError("Las pruebas guardadas no siguen el orden esperado; revisar antes de continuar.")
            restantes = tareas[len(previos):]
            print("Ensayos por ejecutar:", len(restantes), "de", len(tareas), flush=True)
            for nombre, posiciones, es_consulta in restantes:
                textos = [por_query[nombre]["consulta_propuesta"]] if es_consulta else [resumenes[i] for i in posiciones]
                print("Ejecutando:", nombre, "| Textos:", len(textos), flush=True)
                v, datos = codificar(textos, consulta=es_consulta, lote=LOTE)
                datos.update({"prueba": nombre, "fecha_utc": ahora(), "forma": list(v.shape),
                              "article_ids": [article_ids[i] for i in posiciones],
                              "consulta": textos[0] if es_consulta else None, "memoria": memoria()})
                piloto["ensayos"].append(datos)
                guardar_estado(meta, matriz)
                print("  Tiempo:", round(datos["encode_s"], 3), "s | Tokens:", datos["tokens"],
                      "| Dimensiones:", v.shape, flush=True)
            # Sin límite temporal: un ensayo terminado puede durar más de 30 s.
            piloto["estado"] = "completo"
            piloto["fin_utc"] = ahora()
            guardar_estado(meta, matriz)
            print("PILOTO:", piloto["estado"], "| Ensayos:", len(piloto["ensayos"]), "/", len(tareas))
            print("Estado de calidad: NO evaluado. Aún no hay índice del catálogo.")
        except (Exception, KeyboardInterrupt) as error:
            piloto["estado"] = "error_o_interrumpido"
            piloto["fin_utc"] = ahora()
            registrar_error(meta, matriz, "piloto", error)
            raise

Reanudando piloto pausado: ensayos guardados: 2
Cargando Qwen3-VL-Embedding-2B; no se envían imágenes ni vídeo...


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Default prompt name is set to 'default'. This prompt applies unless overridden by `prompt`, `prompt_name`, or a method-specific prompt, including an empty prompt.


Pooling verificado: ['lasttoken'] | include_prompt: True
Modelo: cuda | BF16 solicitado | Carga: 5.561 s
Parámetros por dtype: {'torch.bfloat16': 2127532032}
Parámetros con nombre visual conservados: 406957056
Tokens de documentos mín./máx. (incluida plantilla): 50 620 | Truncados: 0
Instrucción oficial para ambos tipos de texto: "Represent the user's input."
Ensayos por ejecutar: 5 de 7
Ejecutando: mediano | Textos: 1


## 4. Catálogo completo, con reanudación
Activación separada: `EJECUTAR_CATALOGO=True`, solo tras completar y revisar el piloto.

In [ ]:
EJECUTAR_CATALOGO = True

if not EJECUTAR_CATALOGO:
    print("Catálogo no iniciado. Revisa primero las siete llamadas del piloto.")
else:
    comprobar_entradas()
    meta, matriz = cargar_estado()
    if len(matriz) == len(article_ids):
        finalizar_indice(meta, matriz)  # Solo verifica o termina el guardado; no vuelve a codificar.
    else:
        if not meta["pilotos"] or meta["pilotos"][-1]["estado"] != "completo":
            raise RuntimeError("El piloto no terminó correctamente. No se procesa el catálogo todavía.")
        if meta["pilotos"][-1]["entorno"] != ENTORNO:
            raise ValueError("El entorno cambió después del piloto; revisar antes de generar el catálogo.")
        if meta["entorno_vectorizacion"] is not None and meta["entorno_vectorizacion"] != ENTORNO:
            raise ValueError("Cambió el entorno desde los avances guardados. No se mezclan ejecuciones.")
        registrar_ejecucion_sin_pausas(meta)
        sesion = {"id": str(uuid4()), "inicio_utc": ahora(), "filas_iniciales": len(matriz),
                  "estado": "en_curso", "entorno": ENTORNO, "revision_codigo": REVISION_CODIGO}
        meta["sesiones_catalogo"].append(sesion)
        inicio_sesion = perf_counter()
        try:
            asegurar_modelo(meta)
            meta["entorno_vectorizacion"] = ENTORNO
            meta["estado"] = "vectorizacion_incompleta"
            guardar_estado(meta, matriz)
            while len(matriz) < len(article_ids):
                comprobar_entradas()
                inicio = len(matriz)
                fin = min(inicio + LOTE, len(article_ids))
                print(f"Procesando filas {inicio+1}-{fin}/{len(article_ids)} | IDs: {article_ids[inicio:fin]}", flush=True)
                v, datos = codificar(resumenes[inicio:fin], consulta=False, lote=LOTE)
                matriz = np.concatenate([matriz, v], axis=0)
                datos.update({"inicio_fila": inicio, "fin_fila_exclusivo": fin,
                              "article_ids": article_ids[inicio:fin], "fecha_utc": ahora(),
                              "sesion_id": sesion["id"], "memoria_despues": memoria()})
                meta["lotes"].append(datos)
                guardar_estado(meta, matriz)
                print(f"  Guardadas {fin}/{len(article_ids)} | encode: {datos['encode_s']:.3f} s", flush=True)
            # Sin pausa temporal: cada llamada completada guarda un artículo.
            sesion["estado"] = "terminada"
            sesion.update({"fin_utc": ahora(), "filas_finales": len(matriz),
                           "tiempo_sesion_antes_cierre_s": perf_counter()-inicio_sesion,
                           "memoria_al_fin": memoria()})
            guardar_estado(meta, matriz)
            if len(matriz) == len(article_ids):
                finalizar_indice(meta, matriz)
        except (Exception, KeyboardInterrupt) as error:
            sesion.update({"estado": "error_o_interrumpida", "fin_utc": ahora(),
                           "filas_finales": len(matriz), "tiempo_sesion_antes_cierre_s": perf_counter()-inicio_sesion})
            registrar_error(meta, matriz, "catalogo", error)
            raise

## 5. Búsqueda ilustrativa
Activación separada: `EJECUTAR_BUSQUEDA=True`, solo con índice completo.

In [ ]:
EJECUTAR_BUSQUEDA = False

if not EJECUTAR_BUSQUEDA:
    print("Búsqueda no ejecutada. Requiere NPZ completo e índice verificado.")
else:
    comprobar_entradas()
    meta, matriz = cargar_estado()
    if meta["estado"] != "completo" or len(matriz) != len(article_ids) or not INDICE.is_file():
        raise RuntimeError("Falta terminar y verificar el índice; no se busca con un NPZ parcial.")
    if meta["entorno_vectorizacion"] != ENTORNO:
        raise ValueError("El entorno de consulta cambió respecto al índice. Revisar antes de continuar.")
    idx = faiss.read_index(str(INDICE))
    if (idx.ntotal != len(article_ids) or idx.d != DIMENSION
            or idx.metric_type != faiss.METRIC_INNER_PRODUCT
            or sha(INDICE) != meta["sha256_indice"]
            or not np.array_equal(idx.reconstruct_n(0, len(matriz)), matriz)):
        raise ValueError("El índice no coincide con el NPZ.")
    try:
        asegurar_modelo(meta)
        q = por_query["N01-ES"]["consulta_propuesta"]
        vq, datos = codificar([q], consulta=True, lote=1)
        inicio = perf_counter()
        puntuaciones, posiciones = idx.search(vq, min(5, idx.ntotal))
        busqueda_s = perf_counter()-inicio
        resultados = [{"article_id": article_ids[p], "coseno": float(s), "titulo": articulos[p]["titulo"]}
                      for p, s in zip(posiciones[0].tolist(), puntuaciones[0].tolist()) if p >= 0]
        meta["busquedas"].append({"fecha_utc": ahora(), "query_id": "N01-ES", "consulta": q,
                                 "sha256_consultas": SHA_CONSULTAS,
                                 "encode_consulta": datos, "vector_consulta_float32": vq[0].tolist(),
                                 "busqueda_faiss_s": busqueda_s,
                                 "resultados": resultados, "evaluacion_pertinencia": "no_realizada", "revision_codigo": REVISION_CODIGO})
        guardar_estado(meta, matriz)
        print("Consulta:", q)
        print("Codificación de consulta:", round(datos["encode_s"], 3), "s")
        print("Búsqueda FAISS:", round(busqueda_s, 6), "s")
        print("Cinco resultados SIN calificación de pertinencia:")
        for r in resultados:
            print(f"article_id={r['article_id']} | coseno={r['coseno']:.5f} | {r['titulo'][:100]}")
        print("SHA-256 NPZ actualizado:", sha(VECTORES))
        print("SHA-256 FAISS:", sha(INDICE))
    except (Exception, KeyboardInterrupt) as error:
        registrar_error(meta, matriz, "busqueda", error)
        raise